In [0]:
#Parametros de carga
dbutils.widgets.text("tipo_carga","Incremental")
dbutils.widgets.text("fecha_carga", "2025-06-30")

tipo_carga = dbutils.widgets.get("tipo_carga")
fecha_carga =  dbutils.widgets.get("fecha_carga")

In [0]:
#from pyspark.sql.functions import col, sum, lit, current_timestamp

In [0]:
#df_compras =  spark.table("linio.silver_compras")
#df_detalles = spark.table("linio.silver_detalles")

In [0]:
#df_fact_compras =( df_compras.alias("b")
#                    .join(df_detalles.alias("c"), "factura", "inner")
#                    )

#df_fact_compras = df_fact_compras.select(
#   col("b.periodo").alias("periodo"),
#    col("b.venta_id").alias("venta_id"),
#    col("b.factura").alias("factura"),
#    col("b.tipo_compra").alias("tipo_compra"),
#    col("b.fecha_orden").alias("fecha_orden"),
#    col("b.fecha_entrega").alias("fecha_entrega"),
#    col("b.fecha_envio").alias("fecha_envio"),
#    col("b.estado").alias("estado"),
#    col("b.cliente_id").alias("cliente_id"),
#    col("b.vendedor").alias("vendedor"),
#   col("b.departamento").alias("departamento"),
#    col("b.metodo_pago").alias("metodo_pago"),
#    col("b.grupo_dias_envio").alias("grupo_dias_envio"),
#    col("c.detalle_id").alias("detalle_id"),
#    col("c.producto_id").alias("producto_id"),
#    col("c.unidades").alias("unidades"),
#    col("c.Subtotal").alias("Subtotal"),
#    current_timestamp().alias("fecha_actualizacion")
#)

In [0]:
#df_fact_compras.write.mode("overwrite").format("delta").saveAsTable("linio.gold_fact_compras")

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable


In [0]:
if tipo_carga == "Historico":
      # Leer todas las tablas Silver
    df_compras = spark.table("linio.silver_compras")
    df_detalles = spark.table("linio.silver_detalles")

    # Join
    df_fact_compras = (
                     df_compras.alias("b")
                    .join(df_detalles.alias("c"), "factura", "inner")
                    
    )
    df_fact_compras = df_fact_compras.select(
    col("b.periodo").alias("periodo"),
    col("b.venta_id").alias("venta_id"),
    col("b.factura").alias("factura"),
    col("b.tipo_compra").alias("tipo_compra"),
    col("b.fecha_orden").alias("fecha_orden"),
    col("b.fecha_entrega").alias("fecha_entrega"),
    col("b.fecha_envio").alias("fecha_envio"),
    col("b.estado").alias("estado"),
    col("b.cliente_id").alias("cliente_id"),
    col("b.vendedor").alias("vendedor"),
    col("b.departamento").alias("departamento"),
    col("b.metodo_pago").alias("metodo_pago"),
    col("b.grupo_dias_envio").alias("grupo_dias_envio"),
    col("c.detalle_id").alias("detalle_id"),
    col("c.producto_id").alias("producto_id"),
    col("c.unidades").alias("unidades"),
    col("c.Subtotal").alias("Subtotal"),
    current_timestamp().alias("fecha_actualizacion")
    )
    # Carga completa
    (df_fact_compras.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("linio.gold_fact_compras"))
    

else:
   # Obtener los tres meses
    fecha = datetime.strptime(fecha_carga, "%Y-%m-%d")

    particiones = [
        (fecha - relativedelta(months=2)).replace(day=1),
        (fecha - relativedelta(months=1)).replace(day=1),
        fecha.replace(day=1)
    ]

    particiones = [
        p.strftime("%Y-%m-%d")
        for p in particiones
    ]

    # Leer detalles completos
    df_detalles = spark.table("linio.silver_detalles")

    # Leer únicamente las compras de las tres particiones
    df_compras = (
        spark.table("linio.silver_compras").filter(col("periodo").isin(particiones))
    )

    # Join
    df_fact_compras = (
        df_compras.alias("b")
        .join(
            df_detalles.alias("c"),
            "factura",
            "inner"
        )
    )

    df_fact_compras = df_fact_compras.select(
        col("b.periodo").alias("periodo"),
        col("b.venta_id").alias("venta_id"),
        col("b.factura").alias("factura"),
        col("b.tipo_compra").alias("tipo_compra"),
        col("b.fecha_orden").alias("fecha_orden"),
        col("b.fecha_entrega").alias("fecha_entrega"),
        col("b.fecha_envio").alias("fecha_envio"),
        col("b.estado").alias("estado"),
        col("b.cliente_id").alias("cliente_id"),
        col("b.vendedor").alias("vendedor"),
        col("b.departamento").alias("departamento"),
        col("b.metodo_pago").alias("metodo_pago"),
        col("b.grupo_dias_envio").alias("grupo_dias_envio"),
        col("c.detalle_id").alias("detalle_id"),
        col("c.producto_id").alias("producto_id"),
        col("c.unidades").alias("unidades"),
        col("c.Subtotal").alias("Subtotal"),
        current_timestamp().alias("fecha_actualizacion")
    )


    # Merge usando detalle_id
    delta_gold = DeltaTable.forName(spark,"linio.gold_fact_compras")

    delta_gold.alias("gold").merge(df_fact_compras.alias("new"),"gold.detalle_id = new.detalle_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    # Reemplazar únicamente las tres particiones
    (df_fact_compras.write
        .mode("overwrite")
        .format("delta")
        .option("replaceWhere","periodo IN (" +",".join([f"'{p}'" for p in particiones]) +")")
        .saveAsTable("linio.gold_fact_compras")
    )